# Acoustic ID — Module 4: Challan Decision Engine
### Acoustic ID-Based Vehicle Horn Detection for Urban Noise Pollution Control

---

**Scope of this module:**
- Reads `last_decode_status` and `context_flag` from `acoustic_id.db` (written by Module 3)
- Applies the enforcement decision tree to determine challan type and fine amount
- Checks 15-minute deduplication — same vehicle in same zone within 15 minutes attaches to existing challan
- Checks 24-hour repeat offence — doubles the fine for `ACOUSTIC_ID` repeat violations
- Writes challan records to the `challans` table in `acoustic_id.db`
- Exposes `process_all_pending()` to batch-process all vehicles with unread decode results

**Decision tree:**

| Detection Mode | Context Flag | Status | Fine |
|---|---|---|---|
| `ACOUSTIC_ID` | `NORMAL` | `issued` | ₹1,000 (₹2,000 if repeat within 24h) |
| `ACOUSTIC_ID` | `CONTEXTUAL_REVIEW` | `held_for_review` | ₹1,000 |
| `ANPR_FALLBACK` | any | `issued` | ₹5,000 |
| `UNREGISTERED_MODULE` | any | `issued` | ₹5,000 |
| `ACOUSTIC_ID_CORRUPTED` | any | `held_for_review` | ₹1,000 |
| `ERROR` | any | no challan | — |

**Note:** Parivahan portal integration and SMS notification are government deployment concerns.
This module writes to the database only — the challan record is the enforcement artifact.

## Step 1: Import Libraries

In [ ]:
import sqlite3
import os
import uuid
from datetime import datetime, timedelta

## Step 2: Configuration

**Change `DB_FILE`, `ZONE_ID`, and GPS coordinates between deployments.**
All fine amounts and time windows are policy constants — change only if the
Motor Vehicles Act schedule is updated.

| Constant | Value | Source |
|---|---|---|
| `FINE_ACOUSTIC_ID` | ₹1,000 | Section 194F, first offence |
| `FINE_ACOUSTIC_REPEAT` | ₹2,000 | Section 194F, repeat offence |
| `FINE_TAMPER` | ₹5,000 | Tampering / unregistered module |
| `DEDUP_WINDOW_MIN` | 15 | Same vehicle + zone within 15 min → attach to existing challan |
| `REPEAT_WINDOW_HOURS` | 24 | Prior challan within 24h → repeat offence rate |

In [ ]:
# ── USER CONFIGURATION ───────────────────────────────────────────────────────
DB_FILE      = "acoustic_id.db"       # Module 1 database
ZONE_ID      = "ZONE_HOSPITAL_001"    # Silent zone identifier for this deployment
GPS_LAT      = 18.5204                 # Roadside unit GPS latitude
GPS_LON      = 73.8567                 # Roadside unit GPS longitude
# ─────────────────────────────────────────────────────────────────────────────

# Fine amounts — Section 194F, Motor Vehicles Act
FINE_ACOUSTIC_ID     = 1000
FINE_ACOUSTIC_REPEAT = 2000
FINE_TAMPER          = 5000
FINE_UNREGISTERED    = 5000

# Enforcement windows
DEDUP_WINDOW_MIN    = 15   # minutes
REPEAT_WINDOW_HOURS = 24   # hours

notebook_dir = os.getcwd()
db_path      = os.path.join(notebook_dir, DB_FILE)

print(f"Database : '{db_path}'")
print(f"Zone     : {ZONE_ID}")
print(f"GPS      : {GPS_LAT}, {GPS_LON}")

## Step 3: Database Connection and Challans Table

`setup_module4()` opens `acoustic_id.db` and creates the `challans` table
if it does not already exist. Safe to run multiple times.

| Column | Type | Description |
|---|---|---|
| `challan_id` | TEXT PK | UUID generated at issuance |
| `plate_number` | TEXT FK | Vehicle registration number |
| `detection_mode` | TEXT | Copied from Module 3 decode result |
| `context_flag` | TEXT | `NORMAL` or `CONTEXTUAL_REVIEW` |
| `fine_amount` | INTEGER | In rupees |
| `status` | TEXT | `issued` / `held_for_review` / `disputed` / `paid` / `cancelled` |
| `zone_id` | TEXT | Silent zone where violation occurred |
| `gps_lat` / `gps_lon` | REAL | Roadside unit coordinates |
| `issued_at` | TEXT | ISO timestamp of issuance |
| `offence_count` | INTEGER | 1 = first offence, 2 = repeat within 24h |
| `evidence_ref` | TEXT | `.wav` filename that triggered the challan |
| `notes` | TEXT | System-generated notes (e.g. `additional_event` on dedup) |

In [ ]:
def setup_module4(db_path: str) -> sqlite3.Connection:
    """
    Open acoustic_id.db and create the challans table.

    Args:
        db_path (str): Path to acoustic_id.db

    Returns:
        sqlite3.Connection: Open connection with row_factory = sqlite3.Row

    Raises:
        FileNotFoundError: If database does not exist
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError(
            f"Database not found: '{db_path}'\n"
            "Run Module 1 first to generate acoustic_id.db"
        )

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    conn.execute("""
        CREATE TABLE IF NOT EXISTS challans (
            challan_id      TEXT PRIMARY KEY,
            plate_number    TEXT NOT NULL,
            detection_mode  TEXT NOT NULL,
            context_flag    TEXT NOT NULL,
            fine_amount     INTEGER NOT NULL,
            status          TEXT NOT NULL DEFAULT 'issued'
                            CHECK(status IN
                                ('issued','held_for_review','disputed','paid','cancelled')),
            zone_id         TEXT NOT NULL,
            gps_lat         REAL NOT NULL,
            gps_lon         REAL NOT NULL,
            issued_at       TEXT NOT NULL,
            offence_count   INTEGER NOT NULL DEFAULT 1,
            evidence_ref    TEXT DEFAULT NULL,
            notes           TEXT DEFAULT NULL,
            FOREIGN KEY (plate_number) REFERENCES vehicles(plate_number)
        )
    """)
    conn.commit()
    return conn


conn = setup_module4(db_path)
print(f"Database ready: '{db_path}'")

## Step 4: 15-Minute Deduplication

Before issuing a new challan, check if this vehicle was already challan'd
in the same zone within the last 15 minutes.

**Why this matters:** A vehicle might press the horn twice in quick succession
(stalled traffic, near-miss). Without deduplication the driver receives two
challans for what is effectively one incident. The second event is attached
to the existing challan as additional evidence.

In [ ]:
def is_duplicate(
    plate       : str,
    zone_id     : str,
    conn        : sqlite3.Connection,
    window_min  : int = DEDUP_WINDOW_MIN
) -> dict | None:
    """
    Check if a challan already exists for this plate in this zone
    within the deduplication window.

    Args:
        plate      (str): Vehicle plate number
        zone_id    (str): Silent zone identifier
        conn       (sqlite3.Connection): Active DB connection
        window_min (int): Lookback window in minutes

    Returns:
        dict: Existing challan record if found, or None
    """
    cutoff = (datetime.now() - timedelta(minutes=window_min)).isoformat()
    row    = conn.execute(
        """
        SELECT challan_id, fine_amount, status
        FROM   challans
        WHERE  plate_number = ?
          AND  zone_id      = ?
          AND  issued_at    > ?
        ORDER  BY issued_at DESC
        LIMIT  1
        """,
        (plate, zone_id, cutoff)
    ).fetchone()
    return dict(row) if row else None

## Step 5: 24-Hour Repeat Offence Check

If the vehicle already has any challan in the last 24 hours, this event
is treated as a repeat offence. Fine doubles from ₹1,000 to ₹2,000
for `ACOUSTIC_ID` path only. Tamper and unregistered fines are fixed at
₹5,000 regardless of repeat status.

In [ ]:
def is_repeat_offender(
    plate        : str,
    conn         : sqlite3.Connection,
    window_hours : int = REPEAT_WINDOW_HOURS
) -> bool:
    """
    Check if this vehicle has a prior challan within the repeat window.

    Args:
        plate        (str): Vehicle plate number
        conn         (sqlite3.Connection): Active DB connection
        window_hours (int): Lookback window in hours

    Returns:
        bool: True if a prior challan exists within the window
    """
    cutoff = (datetime.now() - timedelta(hours=window_hours)).isoformat()
    row    = conn.execute(
        "SELECT COUNT(*) AS cnt FROM challans WHERE plate_number = ? AND issued_at > ?",
        (plate, cutoff)
    ).fetchone()
    return row["cnt"] > 0

## Step 6: Decision Tree

`decide_action()` is a pure function — it reads the database for repeat check
but writes nothing. Returns an action dict that `issue_challan()` uses to
construct the final record.

Separating the decision from the write makes the logic independently testable
and ensures no partial writes occur if a decision is later reconsidered.

In [ ]:
def decide_action(
    detection_mode : str,
    context_flag   : str,
    plate          : str,
    zone_id        : str,
    conn           : sqlite3.Connection
) -> dict:
    """
    Apply the enforcement decision tree.

    Pure function — reads DB for repeat check but makes no writes.

    Args:
        detection_mode (str): Module 3 decode outcome
        context_flag   (str): "NORMAL" or "CONTEXTUAL_REVIEW"
        plate          (str): Vehicle plate number
        zone_id        (str): Silent zone identifier
        conn           (sqlite3.Connection): Active DB connection

    Returns:
        dict: {
            "action" : "challan" | "no_challan",
            "reason" : str,
            "fine"   : int,
            "status" : "issued" | "held_for_review" | None
        }
    """
    if detection_mode == "ERROR":
        return {"action": "no_challan", "reason": "decode_error",
                "fine": 0, "status": None}

    if detection_mode == "ACOUSTIC_ID":
        if context_flag == "CONTEXTUAL_REVIEW":
            return {"action": "challan", "reason": "acoustic_id_contextual",
                    "fine": FINE_ACOUSTIC_ID, "status": "held_for_review"}
        fine = FINE_ACOUSTIC_REPEAT if is_repeat_offender(plate, conn) else FINE_ACOUSTIC_ID
        return {"action": "challan", "reason": "acoustic_id_normal",
                "fine": fine, "status": "issued"}

    if detection_mode == "ANPR_FALLBACK":
        return {"action": "challan", "reason": "tamper_detected",
                "fine": FINE_TAMPER, "status": "issued"}

    if detection_mode == "UNREGISTERED_MODULE":
        return {"action": "challan", "reason": "unregistered_module",
                "fine": FINE_UNREGISTERED, "status": "issued"}

    if detection_mode == "ACOUSTIC_ID_CORRUPTED":
        return {"action": "challan", "reason": "corrupted_signal",
                "fine": FINE_ACOUSTIC_ID, "status": "held_for_review"}

    # Any other mode — no action
    return {"action": "no_challan", "reason": f"unhandled_mode: {detection_mode}",
            "fine": 0, "status": None}

## Step 7: Issue Challan

`issue_challan()` is the single entry point for challan generation.
It runs deduplication first, then the decision tree, then writes to the DB.

**Return values:**
- `result = 'challan_issued'` — new record written
- `result = 'deduplicated'` — existing challan updated with new evidence
- `result = 'no_challan'` — decode error or unhandled mode, nothing written

In [ ]:
def issue_challan(
    plate          : str,
    detection_mode : str,
    context_flag   : str,
    zone_id        : str,
    gps_lat        : float,
    gps_lon        : float,
    evidence_ref   : str,
    conn           : sqlite3.Connection
) -> dict:
    """
    Full challan issuance pipeline:
    deduplication → decision tree → repeat check → DB write.

    Args:
        plate          (str): Vehicle plate number
        detection_mode (str): Module 3 decode outcome
        context_flag   (str): "NORMAL" or "CONTEXTUAL_REVIEW"
        zone_id        (str): Silent zone identifier
        gps_lat        (float): Violation GPS latitude
        gps_lon        (float): Violation GPS longitude
        evidence_ref   (str): Source .wav filename
        conn           (sqlite3.Connection): Active DB connection

    Returns:
        dict: {
            "result"        : "challan_issued" | "deduplicated" | "no_challan",
            "challan_id"    : str (if issued or deduplicated),
            "plate"         : str,
            "fine"          : int (if issued),
            "status"        : str (if issued),
            "offence_count" : int (if issued),
            "reason"        : str (if no_challan)
        }
    """
    # ── Step 1: Deduplication check ──────────────────────────────────────────
    existing = is_duplicate(plate, zone_id, conn)
    if existing:
        # Attach new evidence to existing challan
        new_evidence = (
            existing.get("evidence_ref", "") + "," + (evidence_ref or "")
        ).strip(",")
        conn.execute(
            """
            UPDATE challans
            SET evidence_ref = ?,
                notes        = 'additional_event'
            WHERE challan_id = ?
            """,
            (new_evidence, existing["challan_id"])
        )
        conn.commit()
        return {
            "result"     : "deduplicated",
            "challan_id" : existing["challan_id"],
            "plate"      : plate
        }

    # ── Step 2: Decision tree ─────────────────────────────────────────────────
    decision = decide_action(detection_mode, context_flag, plate, zone_id, conn)

    if decision["action"] == "no_challan":
        return {"result": "no_challan", "reason": decision["reason"], "plate": plate}

    # ── Step 3: Repeat offence count ──────────────────────────────────────────
    offence_count = 2 if is_repeat_offender(plate, conn) else 1

    # ── Step 4: Write challan record ──────────────────────────────────────────
    challan_id = str(uuid.uuid4())
    conn.execute(
        """
        INSERT INTO challans
            (challan_id, plate_number, detection_mode, context_flag,
             fine_amount, status, zone_id, gps_lat, gps_lon,
             issued_at, offence_count, evidence_ref)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            challan_id, plate, detection_mode, context_flag,
            decision["fine"], decision["status"], zone_id,
            gps_lat, gps_lon,
            datetime.now().isoformat(timespec="seconds"),
            offence_count, evidence_ref
        )
    )
    conn.commit()

    return {
        "result"        : "challan_issued",
        "challan_id"    : challan_id,
        "plate"         : plate,
        "fine"          : decision["fine"],
        "status"        : decision["status"],
        "offence_count" : offence_count
    }

## Step 8: Process All Pending Decode Results

`process_all_pending()` reads every vehicle in the database that has a
`last_decode_status` written by Module 3, and calls `issue_challan()`
for each one. This is the batch runner that connects Module 3 and Module 4.

**Idempotency:** Vehicles processed within the last 15 minutes in the same
zone are deduplicated automatically — re-running this function on the same
decode results will not create duplicate challans.

In [ ]:
def process_all_pending(
    conn        : sqlite3.Connection,
    zone_id     : str,
    gps_lat     : float,
    gps_lon     : float
) -> list:
    """
    Process all vehicles with an unread Module 3 decode result.

    Reads last_decode_status and context_flag from the vehicles table
    and calls issue_challan() for each.

    Args:
        conn    (sqlite3.Connection): Active DB connection
        zone_id (str): Silent zone identifier
        gps_lat (float): Roadside unit GPS latitude
        gps_lon (float): Roadside unit GPS longitude

    Returns:
        list of dict: One result dict per vehicle processed
    """
    rows = conn.execute(
        """
        SELECT plate_number, last_decode_status, context_flag
        FROM   vehicles
        WHERE  last_decode_status IS NOT NULL
        ORDER  BY last_decode_time ASC
        """
    ).fetchall()

    if not rows:
        print("No pending decode results found.")
        print("Run Module 3 first to populate last_decode_status.")
        return []

    results = []
    for row in rows:
        plate   = row["plate_number"]
        mode    = row["last_decode_status"]
        ctx     = row["context_flag"] or "NORMAL"
        # evidence_ref: in production this would be the actual .wav path
        evidence = f"{plate}_latest_decode.wav"

        result = issue_challan(
            plate          = plate,
            detection_mode = mode,
            context_flag   = ctx,
            zone_id        = zone_id,
            gps_lat        = gps_lat,
            gps_lon        = gps_lon,
            evidence_ref   = evidence,
            conn           = conn
        )
        results.append(result)

    return results

## Step 9: Run

Process all pending decode results and print a summary.

In [ ]:
results = process_all_pending(conn, ZONE_ID, GPS_LAT, GPS_LON)

if results:
    issued        = [r for r in results if r["result"] == "challan_issued"]
    deduplicated  = [r for r in results if r["result"] == "deduplicated"]
    no_action     = [r for r in results if r["result"] == "no_challan"]

    print(f"Processed        : {len(results)}")
    print(f"Challans issued  : {len(issued)}")
    print(f"Deduplicated     : {len(deduplicated)}")
    print(f"No action        : {len(no_action)}")

    if issued:
        print()
        hdr = f"  {'Plate':<14}  {'Fine':>7}  {'Status':<18}  {'Offences':>9}  Challan ID"
        print(hdr)
        print(f"  {'-'*14}  {'-'*7}  {'-'*18}  {'-'*9}  {'-'*36}")
        for r in issued:
            print(
                f"  {r['plate']:<14}"
                f"  Rs.{r['fine']:>4}"
                f"  {r['status']:<18}"
                f"  {r['offence_count']:>9}"
                f"  {r['challan_id']}"
            )

    if no_action:
        print()
        print("No action taken:")
        for r in no_action:
            print(f"  {r['plate']} — {r['reason']}")

## Step 10: Challan Viewer

Utility function for inspection. Call manually to review all challan records.
Not part of the enforcement pipeline.

In [ ]:
def show_challans(conn: sqlite3.Connection) -> None:
    """
    Print all challan records in tabular format.

    Args:
        conn (sqlite3.Connection): Active DB connection
    """
    rows = conn.execute(
        """
        SELECT c.challan_id, c.plate_number, v.owner_name,
               c.fine_amount, c.status, c.detection_mode,
               c.context_flag, c.offence_count, c.issued_at, c.zone_id
        FROM   challans c
        JOIN   vehicles v ON c.plate_number = v.plate_number
        ORDER  BY c.issued_at ASC
        """
    ).fetchall()

    if not rows:
        print("No challans issued yet.")
        return

    print(f"CHALLAN RECORDS — {len(rows)} total")
    print("=" * 110)
    hdr = (
        f"{'#':<3}  {'Plate':<14}  {'Owner':<18}  {'Fine':>7}  "
        f"{'Status':<18}  {'Mode':<22}  {'Offences':>9}  Zone"
    )
    print(hdr)
    print("-" * 110)

    for i, row in enumerate(rows, 1):
        print(
            f"{i:<3}  {row[1]:<14}  {row[2][:18]:<18}  "
            f"Rs.{row[3]:>4}  {row[4]:<18}  {row[5]:<22}  "
            f"{row[7]:>9}  {row[9]}"
        )

    # Summary
    total_fines   = sum(r[3] for r in rows)
    issued_count  = sum(1 for r in rows if r[4] == "issued")
    held_count    = sum(1 for r in rows if r[4] == "held_for_review")
    print()
    print(f"Total fines      : Rs.{total_fines:,}")
    print(f"Issued           : {issued_count}")
    print(f"Held for review  : {held_count}")

## Step 11: Close Database Connection

In [ ]:
conn.close()